<div align="center" style="font-size:28px; font-weight:bold;">
MFE 409: Financial Risk Management
<br>Problem Set 6
<br>Valentin Haddad
<br>Due 2/26 before midnight
</div>

<br>

Cohort 2, Group 6:
Lee James, Moazzami Ali, Yu Aiden, Cai Jenny, Li Zehao, Guo Lucy

## 1 Bootstrapping a CDS curve

#### 1. Recover the hazard rate curve from the slide “Bootstrapping Default Probabilities from CDS” (slide 17) of the notes.

In [7]:
import numpy as np
from scipy.optimize import fsolve

t = [3, 5, 10]
spreads = [0.0050, 0.0060, 0.0100]
LGD = 0.40

lambdas = []

for i in range(len(t)):

    def objective(L):
        test_lambdas = lambdas + [L[0]]

        premium_leg = 0
        cum_hazard = 0
        t_prev = 0

        for j in range(i + 1):
            lam = test_lambdas[j]
            dt = t[j] - t_prev

            segment_val = dt if lam == 0 else (1 - np.exp(-lam * dt)) / lam
            premium_leg += np.exp(-cum_hazard) * segment_val

            cum_hazard += lam * dt
            t_prev = t[j]

        default_leg = 1 - np.exp(-cum_hazard)

        calc_spread = LGD * default_leg / premium_leg
        return calc_spread - spreads[i]

    guess = spreads[i] / LGD

    res = fsolve(objective, [guess])[0]
    lambdas.append(res)

for time, lam in zip(t, lambdas):
    print(f"Maturity {time}yr: lambda = {lam:.4f}")

Maturity 3yr: lambda = 0.0125
Maturity 5yr: lambda = 0.0189
Maturity 10yr: lambda = 0.0364


#### 2. Use this hazard rate curve to price a 7-year bond on the same company which pays 3% coupon every 6 month and has face value $100.

In [10]:
import numpy as np

face_value = 100
maturity = 7
recovery_rate = 0.60
coupon_pmt = face_value * (0.03)

lambdas = [0.0125, 0.0189, 0.0364]
t_nodes = [3, 5, 10]

def get_survival_prob(t):
    cum_hazard = 0
    t_prev = 0

    for lam, t_node in zip(lambdas, t_nodes):
        if t > t_node:
            cum_hazard += lam * (t_node - t_prev)
            t_prev = t_node
        else:
            cum_hazard += lam * (t - t_prev)
            break

    return np.exp(-cum_hazard)

times = np.arange(0.5, maturity + 0.5, 0.5)

bond_price = 0
s_prev = 1.0

for t in times:
    s_t = get_survival_prob(t)

    bond_price += coupon_pmt * s_t

    prob_default = s_prev - s_t
    bond_price += (face_value * recovery_rate) * prob_default

    s_prev = s_t

s_final = get_survival_prob(maturity)
bond_price += face_value * s_final

print(f"Survival Probability at Year 7: {s_final:.4f} (or {s_final*100:.2f}%)")
print(f"Fair Price of the Risky Bond: ${bond_price:.2f}")

Survival Probability at Year 7: 0.8623 (or 86.23%)
Fair Price of the Risky Bond: $134.03


## 2 Historical vs bond-implied hazard rates

#### Explain the patterns you see in the table on the slide “Comparing Hazard Rates” (slide 19) of the notes.

#### Solution:

Based on the table, the most prominent pattern is that the implied hazard rates extracted from bond prices are consistently higher than the actual historical default rates across all credit ratings. While both rates predictably increase as credit quality declines from Aaa to Caa, the gap between them reveals that investors demand a significant credit risk premium. Bondholders require compensation not just for the statistically expected losses, but also for bearing the uncertainty of default, along with other factors like liquidity constraints and systemic market risks.

A secondary, but crucial, pattern emerges in how this premium scales across the different ratings. The ratio of implied to historical rates is extraordinarily high for safe, investment-grade bonds and drops steadily for riskier high-yield bonds. Conversely, the absolute difference grows larger as credit quality worsens. This suggests that for highly rated bonds where actual defaults are extremely rare, the credit spread is driven heavily by baseline systemic factors, liquidity premiums, and tax differences rather than pure default expectations, making the implied rate proportionally massive compared to the near-zero historical risk.

## 3 Risk management and regulation after the 2008 Financial Crisis

#### Each study group is assigned to a bank as follows and responsible for summarizing their risk management policies. Your group number can be found in the attached list.

$$
\begin{array}{|c|l|}
\hline
\textbf{Group} & \textbf{Bank} \\ \hline
1/11  & \text{Goldman Sachs}    \\ \hline
2/12  & \text{UBS}              \\ \hline
3/13  & \text{JP Morgan Chase}  \\ \hline
4/14  & \text{Citigroup}        \\ \hline
5/15  & \text{Barclays Capital} \\ \hline
6/16  & \textbf{Morgan Stanley}   \\ \hline
7/17  & \text{Deutsche Bank}    \\ \hline
8/18  & \text{Bank of America}  \\ \hline
9/19  & \text{BNP Paribas}      \\ \hline
10/20 & \text{Credit Suisse}    \\ \hline
\end{array}
$$

#### Download their 2009 and most recent annual reports (10-K for US firms and 20-F or 6-K for foreign firms) from SEC's website (https://www.sec.gov/edgar/searchedgar/companysearch.html). Write a short essay describing the approach of the bank is following for risk management. In particular, describe how it computes the various risk measures to respect the Basel regulations.

#### Solution:

Morgan Stanley's risk management framework relies on independent oversight, comprehensive reporting, and rigorous statistical methodologies to monitor market, credit, and operational risks. To comply with regulatory standards and measure market risk, the firm heavily utilizes Value-at-Risk (VaR) models. Specifically, the company employs a historical simulation approach, leveraging a four-year window of historical data to construct a distribution of potential daily portfolio value changes. This primary VaR metric is computed at a 99% confidence level for a one-day holding period, serving as a foundational statistical tool for daily risk assessment.

Beyond standard market risk, the firm models counterparty credit risk using advanced probabilistic techniques, including Monte Carlo simulations to estimate Potential Future Exposure (PFE). These models simulate various market scenarios to evaluate the expected credit exposure over the life of a transaction. Because historical VaR has limitations in predicting unprecedented market shocks, the firm supplements its quantitative models with comprehensive stress testing. This involves applying extreme but plausible macro-economic and market-specific shocks to the portfolio to assess potential vulnerabilities that standard statistical measures might underestimate.

To align directly with Basel regulations, the firm translates these risk measures into regulatory capital requirements. In response to enhanced Basel Committee market risk frameworks, the bank computes the Incremental Risk Charge (IRC) to capture default and credit migration risks for unsecuritized products. Additionally, it calculates the Comprehensive Risk Measure (CRM) to assess the risk of specific correlation trading portfolios. Together, these modeling approaches ensure the firm maintains adequate capital buffers against complex, multidimensional financial risks.

## 4 Interview questions

#### 1. You are long an option that promises to pay the square of the stock price in three months ($S_{t+3months}^2$). Does the delta approach to computing VaR underestimate or overestimate your risk?

#### Solution:

The delta approach to computing Value at Risk (VaR) will underestimate the risk because it relies on a linear approximation of the option's value, effectively ignoring the convex nature of the payoff. Since the payoff is the square of the stock price ($S^2$), the option has a constant positive gamma. In a long position, this convexity works in our favor by increasing the option's value more quickly as prices rise and decreasing it more slowly as prices fall compared to a linear model. Consequently, the actual distribution of returns is positively skewed, meaning the "fat tail" of potential losses is thinner than a linear delta-normal model would predict. However, in the context of standard risk management, the delta-only approach fails to capture the non-linear acceleration of price changes, leading to an inaccurate and typically underestimated measure of the actual potential loss at a given confidence level.

#### 2. A stock has been going up 10% each year for the past three years. What is the three-month forward price of this stock today?

#### Solution:
The stock's past performance, even if it's been going up 10% annually, is actually irrelevant for determining its three-month forward price today. The forward price is calculated based on the current spot price of the stock, the risk-free interest rate for the three-month period, and any expected dividends.

Specifically, if we assume no dividends, the three-month forward price would be found using the formula: Spot Price * e^(risk-free rate * Time). So, I would need the current spot price of the stock and the three-month risk-free interest rate to provide an exact figure.

#### 3. You are trading quanto options. What are the important sources of risks to include in your risk management model. If you are not familiar with quanto options, you can find information online.

#### Solution:

Risk management for quanto options requires accounting for both the price volatility of the underlying foreign asset and the currency risk, specifically the correlation between the asset price and the exchange rate (the "quanto adjustment"). Beyond standard Greeks like Delta and Gamma for the asset, the model must track FX Delta and Cross-Gamma, as shifts in the exchange rate affect the domestic value of the payoff even though the strike is fixed in domestic terms. Additionally, because the payoff involves two different interest rate environments, you must manage sensitivity to both domestic and foreign interest rate curves (Rho), alongside the volatility smile of both the asset and the currency pair.